<a href="https://colab.research.google.com/github/sahil9022-crypto/Hackthon-project/blob/main/dynamic_clustering_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# =============================
# Install Dependencies
# =============================
!pip install gradio scikit-learn plotly pandas numpy

import gradio as gr
import pandas as pd
import numpy as np
from sklearn.cluster import (
    KMeans, AgglomerativeClustering, DBSCAN,
    SpectralClustering, Birch, MeanShift
)
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import plotly.express as px

# =============================
# Business Insight Generator
# =============================
def generate_business_insights(df, features):
    insights = []
    overall_mean = df[features].mean()

    valid_clusters = [c for c in df["Cluster"].unique() if c != -1]

    for cluster in sorted(valid_clusters):
        cdf = df[df["Cluster"] == cluster]
        size_pct = round(len(cdf) / len(df) * 100, 1)
        cmean = cdf[features].mean()

        high = cmean[cmean > overall_mean].index.tolist()
        low = cmean[cmean < overall_mean].index.tolist()

        action = (
            "Scale aggressively (high value segment)"
            if len(high) >= len(low)
            else "Improve engagement or pricing"
        )

        insights.append(f"""
### 🔹 Cluster {cluster}
- **Population:** {size_pct}%
- **Strengths:** {', '.join(high) if high else 'None'}
- **Weaknesses:** {', '.join(low) if low else 'None'}
- **Recommended Action:** {action}
""")

    if -1 in df["Cluster"].unique():
        noise_pct = round((df["Cluster"] == -1).sum() / len(df) * 100, 1)
        insights.append(f"""
### ⚠ Noise / Outliers
- **Population:** {noise_pct}%
- **Insight:** Irregular or low-engagement records. Consider cleanup or separate strategy.
""")

    return "\n".join(insights)

# =============================
# Core Clustering Logic
# =============================
def run_clustering(file, algo, k, eps, min_samples, enable_3d):
    try:
        df = pd.read_csv(file.name)

        X = df.select_dtypes(include=np.number)
        X = X.dropna()

        if X.shape[1] < 2:
            raise ValueError("CSV must contain at least 2 numeric columns")

        models = {
            "KMeans": KMeans(n_clusters=k, random_state=42),
            "Agglomerative": AgglomerativeClustering(n_clusters=k),
            "DBSCAN": DBSCAN(eps=eps, min_samples=min_samples),
            "GMM": GaussianMixture(n_components=k, random_state=42),
            "Spectral": SpectralClustering(n_clusters=k, random_state=42),
            "BIRCH": Birch(n_clusters=k),
            "MeanShift": MeanShift()
        }

        labels = models[algo].fit_predict(X)
        df = df.loc[X.index].copy()
        df["Cluster"] = labels

        # PCA
        pca = PCA(n_components=3 if enable_3d else 2)
        reduced = pca.fit_transform(X)
        df["PCA1"], df["PCA2"] = reduced[:, 0], reduced[:, 1]
        if enable_3d:
            df["PCA3"] = reduced[:, 2]

        # Visualization
        fig = (
            px.scatter_3d(
                df, x="PCA1", y="PCA2", z="PCA3",
                color=df["Cluster"].astype(str),
                title=f"{algo} Clustering"
            )
            if enable_3d
            else px.scatter(
                df, x="PCA1", y="PCA2",
                color=df["Cluster"].astype(str),
                title=f"{algo} Clustering"
            )
        )

        # Silhouette Score (safe)
        sil = (
            round(silhouette_score(X, labels), 3)
            if len(set(labels)) > 1 and algo not in ["DBSCAN", "MeanShift"]
            else "N/A"
        )

        # Cluster Summary
        cluster_summary = (
            df.groupby("Cluster")[X.columns]
            .mean()
            .round(2)
            .to_html(border=0)
        )

        # Business Insights
        insights = generate_business_insights(df, X.columns)

        summary_md = f"""
## 📊 Clustering Overview
- **Algorithm:** {algo}
- **Clusters Found:** {len(set(labels))}
- **Silhouette Score:** {sil}

## 💡 Business Insights
{insights}
"""

        return (
            summary_md,
            df.head(25).to_html(border=0),
            fig,
            cluster_summary,
            sil
        )

    except Exception as e:
        return f"❌ Error: {e}", None, None, None, None

# =============================
# UI
# =============================
with gr.Blocks(theme="gradio/soft") as app:
    gr.Markdown("## 🧠 Universal Business Clustering Dashboard")

    file = gr.File(label="Upload ANY CSV (numeric columns auto-detected)", type="filepath")

    with gr.Row():
        algo = gr.Radio(
            ["KMeans", "Agglomerative", "DBSCAN", "GMM", "Spectral", "BIRCH", "MeanShift"],
            value="KMeans",
            label="Algorithm"
        )
        k = gr.Slider(2, 12, 4, step=1, label="Clusters")
        eps = gr.Slider(0.1, 10.0, 0.5, step=0.1, label="DBSCAN eps")
        min_samples = gr.Slider(1, 10, 5, step=1, label="DBSCAN min_samples")
        enable_3d = gr.Checkbox(label="3D View")

    run = gr.Button("🚀 Run Analysis")

    with gr.Tab("📊 Summary & Insights"):
        summary_md = gr.Markdown()
        sil_box = gr.Textbox(label="Silhouette Score")

    with gr.Tab("📈 Visualization"):
        plot = gr.Plot()

    with gr.Tab("📋 Top 25 Rows"):
        table = gr.HTML()
        cluster_stats = gr.HTML()

    run.click(
        run_clustering,
        inputs=[file, algo, k, eps, min_samples, enable_3d],
        outputs=[summary_md, table, plot, cluster_stats, sil_box]
    )

app.launch(share=True)


/tmp/ipython-input-2873530438.py:150: DeprecationWarning:

The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.



Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://dbc2cf2cb072c7e8b1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# 🧠 Universal Business Clustering Dashboard

## 📌 Project Overview

This project is a **production-ready, UI-driven clustering analytics dashboard** that allows users to upload **any CSV file**, automatically analyze numeric data, apply **7 different clustering algorithms**, and generate **business-oriented insights** instead of raw ML outputs.

The goal is simple but serious:

> **Convert raw data → clusters → actionable business insights**

This is **not** a toy ML demo. It is designed to be:

* Generic (works with any dataset)
* Explainable (clear insights, not just charts)
* Extendable (API, scaling, deployment-ready)

---

## 🚀 Key Features

* Upload **any CSV file** (no fixed schema required)
* Automatic numeric feature detection
* 7 industry-relevant clustering algorithms
* 2D / 3D PCA-based visualization
* Cluster-wise statistics
* Dynamic **business insights generation**
* Clean, scalable UI using Gradio

---

## 🧰 Tech Stack

### Frontend / UI

* **Gradio** – Rapid, production-friendly UI for ML applications

### Backend / Data Processing

* **Python 3** – Core language
* **Pandas** – Data loading, cleaning, aggregation
* **NumPy** – Numerical operations

### Machine Learning

* **Scikit-learn** – Clustering algorithms, PCA, metrics

### Visualization

* **Plotly** – Interactive 2D & 3D visualizations

---

## 🧠 Clustering Algorithms Used (7 Total)

### 1️⃣ KMeans Clustering

**How it works:**

* Divides data into *K clusters*
* Minimizes distance between points and cluster centroid

**When to use:**

* Well-separated, spherical clusters
* Fast and interpretable

**Limitation:**

* Requires predefined `K`

---

### 2️⃣ Agglomerative (Hierarchical) Clustering

**How it works:**

* Bottom-up approach
* Starts with each point as a cluster and merges them

**When to use:**

* When cluster hierarchy matters
* Small-to-medium datasets

**Limitation:**

* Computationally expensive

---

### 3️⃣ DBSCAN (Density-Based Clustering)

**How it works:**

* Groups points based on density
* Identifies noise/outliers automatically

**When to use:**

* Arbitrary-shaped clusters
* Outlier detection

**Limitation:**

* Sensitive to `eps` and `min_samples`

---

### 4️⃣ Gaussian Mixture Model (GMM)

**How it works:**

* Probabilistic clustering
* Each cluster modeled as a Gaussian distribution

**When to use:**

* Overlapping clusters
* Soft cluster membership

**Limitation:**

* Assumes Gaussian distribution

---

### 5️⃣ Spectral Clustering

**How it works:**

* Uses graph-based similarity
* Performs clustering in a reduced eigen-space

**When to use:**

* Non-linear, complex cluster shapes

**Limitation:**

* Computationally heavy

---

### 6️⃣ BIRCH Clustering

**How it works:**

* Incremental, tree-based clustering
* Designed for large datasets

**When to use:**

* Large-scale, streaming data

**Limitation:**

* Less accurate on complex shapes

---

### 7️⃣ MeanShift Clustering

**How it works:**

* Density-based
* Automatically finds number of clusters

**When to use:**

* No idea about number of clusters

**Limitation:**

* Slow for large datasets

---

## 📊 Dimensionality Reduction (PCA)

### Why PCA?

Most datasets contain many numeric columns. PCA:

* Reduces dimensions to 2D / 3D
* Preserves maximum variance
* Enables visualization

### How it’s used here:

* PCA is applied **only for visualization**
* Clustering is performed on original numeric features

---

## 📈 Evaluation Metric

### Silhouette Score

* Measures how well-separated clusters are
* Range: `-1` to `+1`

**Higher is better**

> Automatically disabled for algorithms where it is not meaningful (DBSCAN, MeanShift)

---

## 💡 Business Insights Generation (Core Feature)

This is what makes the project **valuable**.

### How insights are generated:

For each cluster:

1. Cluster size (% of total data)
2. Mean of each numeric feature
3. Comparison with global dataset mean
4. Feature strengths & weaknesses
5. Action-oriented recommendation

### Example Insights:

* High-value segments → Upsell & premium targeting
* Low-performing clusters → Retention / pricing review
* Noise points → Data cleanup or special handling

Outliers (`Cluster = -1`) are **explicitly analyzed**, not ignored.

---

## 🖥️ Application Workflow

1. Upload CSV file
2. Numeric features auto-detected
3. Select clustering algorithm
4. Adjust parameters (if needed)
5. Run analysis
6. View:

   * Business summary
   * Interactive visualization
   * Top 25 records
   * Cluster statistics

---

## 📦 Project Structure

```
├── app.py            # Main application
├── README.md         # Project documentation
├── requirements.txt  # Dependencies
```

---

## 🚀 Deployment Ready

This project can be deployed on:

* Hugging Face Spaces
* Docker + Cloud Run
* AWS / Azure / GCP

The architecture is already UI + backend cleanly separated.

---

## 🔮 Future Enhancements

* Feature scaling toggle (Standard / MinMax)
* Auto-cluster recommendation (Elbow + Silhouette)
* Insight export (PDF / JSON)
* REST API using FastAPI
* Authentication & role-based dashboards

---

## 👤 Author

**Sahil Pawar**
Second-Year Engineering Student | Data Science & AI Enthusiast

---

## ⚠️ Final Note

This project is intentionally **generic, explainable, and business-focused**.
If you remove the insights layer, it becomes just another clustering demo.

**The insights layer is the product.**
